In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv("../data/raw/parking_sessions.csv")
df.head()

,record_id,location_id,entry_time,exit_time,vehicle_type,recorded_duration,is_season_vehicle,is_normal_vehicle,vehicle_status,parking_slot,duration_seconds,is_valid_timestamp,is_zero_duration,is_positive_duration
0,239854,1,2023-08-07 12:11:38.115638,2023-08-07 18:50:41.155646,2,399.050000,1,1,5,0,23943.040008,True,False,True
1,239878,1,2023-08-07 14:42:38.504172,2023-08-07 15:03:47.749831,1,21.150000,1,1,7,0,1269.245659,True,False,True
2,239904,1,2023-08-07 16:38:00.409214,2023-08-07 17:23:29.422259,1,45.483333,1,1,5,0,2729.013045,True,False,True
3,240057,1,2023-08-08 11:55:46.409214,2023-08-08 11:55:46.409214,2,0.000000,1,1,11,0,0.000000,True,True,False
4,240058,1,2023-08-08 11:56:13.409214,2023-08-08 11:56:13.409214,1,0.000000,1,1,10,0,0.000000,True,True,False


In [3]:
clean_df = df.copy()

clean_df["entry_time"] = pd.to_datetime(clean_df["entry_time"])
clean_df["exit_time"] = pd.to_datetime(clean_df["exit_time"])

clean_df["duration_seconds"] = (
    clean_df["exit_time"] - clean_df["entry_time"]
).dt.total_seconds()

# Identify session types
clean_df["session_status"] = np.select(
    [
        clean_df["duration_seconds"] < 0,
        clean_df["duration_seconds"] == 0,
        clean_df["duration_seconds"] > 0
    ],
    [
        "invalid",
        "active",
        "completed"
    ],
    default="unknown"
)

print(clean_df["session_status"].value_counts())

session_status
completed    204621
active        53390
invalid         416
Name: count, dtype: int64


In [7]:
completed_df = clean_df[clean_df["session_status"] == "completed"]

completed_df['duration_hours'] = completed_df['duration_seconds'] / 3600
completed_df['entry_time'] = pd.to_datetime(completed_df['entry_time'])
completed_df['exit_time'] = pd.to_datetime(completed_df['exit_time'])
completed_df['entry_day'] = completed_df['entry_time'].dt.day
completed_df['entry_month'] = completed_df['entry_time'].dt.month

completed_df['day_of_week'] = completed_df['entry_time'].dt.dayofweek
completed_df['is_weekend'] =(completed_df['day_of_week'] >= 5).astype(int)
completed_df['exit_hour'] = completed_df['exit_time'].dt.hour

C:\Users\ASUS\AppData\Local\Temp\ipykernel_50908\460218734.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  completed_df['duration_hours'] = completed_df['duration_seconds'] / 3600
C:\Users\ASUS\AppData\Local\Temp\ipykernel_50908\460218734.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  completed_df['entry_time'] = pd.to_datetime(completed_df['entry_time'])
C:\Users\ASUS\AppData\Local\Temp\ipykernel_50908\460218734.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a

In [9]:
completed_df.head()

,record_id,location_id,entry_time,exit_time,vehicle_type,recorded_duration,is_season_vehicle,is_normal_vehicle,vehicle_status,parking_slot,...,is_valid_timestamp,is_zero_duration,is_positive_duration,session_status,duration_hours,entry_day,entry_month,day_of_week,is_weekend,exit_hour
0,239854,1,2023-08-07 12:11:38.115638,2023-08-07 18:50:41.155646,2,399.050000,1,1,5,0,...,True,False,True,completed,6.650844,7,8,0,0,18
1,239878,1,2023-08-07 14:42:38.504172,2023-08-07 15:03:47.749831,1,21.150000,1,1,7,0,...,True,False,True,completed,0.352568,7,8,0,0,15
2,239904,1,2023-08-07 16:38:00.409214,2023-08-07 17:23:29.422259,1,45.483333,1,1,5,0,...,True,False,True,completed,0.758059,7,8,0,0,17
5,240110,1,2023-08-08 17:17:31.673766,2023-08-08 18:21:26.912024,2,63.916667,1,1,7,0,...,True,False,True,completed,1.065344,8,8,1,0,18
6,240115,1,2023-08-08 17:54:24.409214,2023-08-08 17:55:02.155484,1,0.616667,1,1,2,0,...,True,False,True,completed,0.010485,8,8,1,0,17


In [10]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

completed_df.to_csv("../data/processed/comepleted_parking_sessions.csv", index=False)